# DenseNet121 EXP-027 · Split (12) → Holdout (7)

Đánh giá độc lập **năm checkpoint đã huấn luyện** của run 027 ở 256 px. Notebook không train lại,
không chọn seed theo Holdout (7), và không đánh giá lại Test. Split (12) được nhận diện bằng
SHA-256 của ZIP và manifest trong gói FULL gốc; không cần tải lại 1,47 GB dữ liệu train.

Chạy lần lượt các cell trên Colab (GPU runtime). Cell tải Holdout chỉ chạy sau khi xác minh
config, Validation lock, manifest và checkpoint. Kết quả nằm trong thư mục/ZIP riêng rồi được
tải lên thư mục Dataset trên Drive. Có thể đặt hai đường dẫn ZIP local trong cell cấu hình.


In [1]:
import base64, gc, hashlib, io, json, os, shutil, sys, time, zipfile, zlib
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from IPython.display import display

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))


TensorFlow 2.21.0 | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Frozen 027 source, no recipe or checkpoint changes.
SOURCE_EXP_ID = 'EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b'
SOURCE_CODE_REVISION = '3e196f0b304c3ef09c97d56d26217abff0a3d450cfe6cfcb3bfcb01713b299e4'
SOURCE_FULL_NAME = SOURCE_EXP_ID + '_FULL.zip'
SOURCE_FULL_ID = '132dkxcnKYfmHE67UyYxqQrW0M7Cqpq0z'
SOURCE_FULL_BYTES = 630000922
SPLIT_NAME = 'CashewData_Split (12).zip'
SPLIT_ID = '1XMNuHP6RFVltsA8fBXcAh7undxzosKYO'
HOLDOUT_NAME = 'RL_Cashew_holdout (7).zip'
HOLDOUT_ID = '1BNFwh95vCTnN7YjLE82s_ipYUzutfabq'
HOLDOUT_BYTES = 370644059
DRIVE_TARGET_FOLDER_ID = '1-Mrqs8Y9dwLexSKIVKBvyYvvQ8X8fXti'
EVAL_ID = SOURCE_EXP_ID + '-HOLDOUT07-EVAL'
SOURCE_FULL_ZIP_OVERRIDE = None  # e.g. '/content/drive/MyDrive/..._FULL.zip'
HOLDOUT_ZIP_OVERRIDE = None      # e.g. '/content/drive/MyDrive/RL_Cashew_holdout (7).zip'
UPLOAD_ANALYSIS_TO_DRIVE = True
WORK_ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
CACHE_DIR = WORK_ROOT / 'densenet121_027_holdout07_cache'
OUT_DIR = WORK_ROOT / 'training_results' / 'current_dataset' / EVAL_ID
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Source:', SOURCE_EXP_ID, '| Evaluation:', EVAL_ID)


Source: EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b | Evaluation: EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b-HOLDOUT07-EVAL


In [3]:
# Direct Drive-ID downloads. Drive API uses pageSize, never page_size.
def get_drive_service(scope='https://www.googleapis.com/auth/drive.readonly'):
    import importlib.util, subprocess
    for package, module in [('google-auth', 'google.auth'),
                            ('google-api-python-client', 'googleapiclient')]:
        try: installed = importlib.util.find_spec(module)
        except ModuleNotFoundError: installed = None
        if installed is None:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
    import google.auth
    from googleapiclient.discovery import build
    if 'google.colab' in sys.modules or Path('/content').is_dir():
        from google.colab import auth
        auth.authenticate_user()
    credentials, _ = google.auth.default(scopes=[scope])
    return build('drive', 'v3', credentials=credentials, cache_discovery=False)

def file_md5(path):
    digest = hashlib.md5()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 << 20), b''): digest.update(block)
    return digest.hexdigest()

def ensure_zip(file_id, name, expected_bytes, override=None):
    if override is not None:
        path = Path(override).expanduser()
        if not path.is_file() or path.stat().st_size != expected_bytes or not zipfile.is_zipfile(path):
            raise ValueError(f'Local ZIP does not match expected name/size: {path}')
        return path
    service = get_drive_service()
    meta = service.files().get(fileId=file_id, fields='id,name,size,md5Checksum',
                               supportsAllDrives=True).execute()
    if meta['id'] != file_id or meta['name'] != name or int(meta['size']) != expected_bytes:
        raise ValueError(f'Drive file differs from frozen source: {meta}')
    destination = CACHE_DIR / name
    if destination.is_file() and destination.stat().st_size == expected_bytes:
        if not meta.get('md5Checksum') or file_md5(destination) == meta['md5Checksum']:
            return destination
    from googleapiclient.http import MediaIoBaseDownload
    temporary = destination.with_suffix('.zip.part')
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with temporary.open('wb') as stream:
        downloader = MediaIoBaseDownload(stream, request, chunksize=8 << 20)
        done = False
        while not done:
            progress, done = downloader.next_chunk(num_retries=3)
            if progress: print(name, f'{progress.progress():.0%}')
    if temporary.stat().st_size != expected_bytes or not zipfile.is_zipfile(temporary):
        raise ValueError(f'Incomplete ZIP: {name}')
    if meta.get('md5Checksum') and file_md5(temporary) != meta['md5Checksum']:
        raise ValueError(f'Drive MD5 mismatch: {name}')
    temporary.replace(destination)
    return destination


## 1. Xác minh run 027 trước khi mở Holdout (7)

Chỉ giải nén config, Validation lock, manifest, kết quả Validation, stage selection và năm
checkpoint từ gói FULL. Mọi hash trong lock phải khớp. Cell sẽ dừng nếu source thiếu hoặc sai.


In [4]:
source_zip = ensure_zip(SOURCE_FULL_ID, SOURCE_FULL_NAME, SOURCE_FULL_BYTES,
                        SOURCE_FULL_ZIP_OVERRIDE)
SOURCE_DIR = CACHE_DIR / SOURCE_EXP_ID
needed = {'experiment_config.json', 'validation_selection_lock.json',
          'dataset_manifest.csv', 'aggregate/validation_results_5seeds.csv'}
for seed in [42, 123, 2026, 3407, 7777]:
    needed.update({f'seed_{seed}/best.weights.h5', f'seed_{seed}/stage_selection.json'})
prefix = SOURCE_EXP_ID + '/'
with zipfile.ZipFile(source_zip) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)): raise ValueError('Duplicate FULL archive members')
    if not needed.issubset({name[len(prefix):] for name in names if name.startswith(prefix)}):
        raise ValueError('FULL archive lacks required frozen-027 files')
    for relative in sorted(needed):
        target = SOURCE_DIR / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        member = archive.getinfo(prefix + relative)
        if member.is_dir() or member.file_size <= 0:
            raise ValueError(f'Invalid source file: {relative}')
        if target.is_file() and target.stat().st_size == member.file_size:
            continue  # Cryptographic checks below also verify cached files.
        temporary = target.with_name(target.name + '.part')
        with archive.open(member) as source_stream, temporary.open('wb') as output:
            shutil.copyfileobj(source_stream, output, 8 << 20)
        if temporary.stat().st_size != member.file_size: raise ValueError(relative)
        temporary.replace(target)

config_path = SOURCE_DIR / 'experiment_config.json'
lock_path = SOURCE_DIR / 'validation_selection_lock.json'
config = json.loads(config_path.read_text(encoding='utf-8'))
selection_lock = json.loads(lock_path.read_text(encoding='utf-8'))
if config['experiment_id'] != SOURCE_EXP_ID or selection_lock['experiment_id'] != SOURCE_EXP_ID:
    raise ValueError('Wrong EXP-027 source')
if config['code_revision_sha256'] != SOURCE_CODE_REVISION:
    raise ValueError('EXP-027 code revision changed')
if config['dataset']['archive'] != SPLIT_NAME or config['dataset']['drive_file_id'] != SPLIT_ID:
    raise ValueError('Source is not frozen Split (12)')
if config['model']['backbone'] != 'DenseNet121' or config['model']['weights'] != 'imagenet':
    raise ValueError('Model recipe changed')
if config['model']['augmentation']['brightness_factor'] != 0.08 or '256x256' not in config['input']:
    raise ValueError('Source is not the 256 px EXP-027 recipe')
SEEDS = [42, 123, 2026, 3407, 7777]
CLASS_NAMES = ['anthracnose', 'healthy', 'leaf_miner', 'red_rust', 'not_cashew_leaf']
if config['seeds'] != SEEDS or config['dataset']['class_names'] != CLASS_NAMES:
    raise ValueError('Seeds or class order changed')
if selection_lock['seeds'] != SEEDS or selection_lock['class_names'] != CLASS_NAMES:
    raise ValueError('Validation lock seeds/classes changed')
if file_sha256(config_path) != selection_lock['config_sha256']:
    raise ValueError('Config SHA-256 differs from Validation lock')
if (config['dataset']['archive_sha256'] != selection_lock['dataset_sha256'] or
        config['dataset']['manifest_sha256'] != selection_lock['dataset_manifest_sha256']):
    raise ValueError('Split (12) fingerprint differs from Validation lock')
manifest = pd.read_csv(SOURCE_DIR / 'dataset_manifest.csv')
if not {'split', 'path', 'relative_path', 'label', 'sha256'}.issubset(manifest.columns):
    raise ValueError('Incomplete Split (12) manifest')
if set(manifest['split']) != {'train', 'val', 'test'} or manifest['sha256'].isna().any():
    raise ValueError('Split (12) manifest missing partitions/hashes')
records = manifest[['relative_path', 'label', 'sha256']].to_dict('records')
manifest_hash = hashlib.sha256(json.dumps(records, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
if manifest_hash != selection_lock['dataset_manifest_sha256']:
    raise ValueError('Split (12) manifest SHA-256 mismatch')
validation_path = SOURCE_DIR / 'aggregate' / 'validation_results_5seeds.csv'
if file_sha256(validation_path) != selection_lock['validation_sha256']:
    raise ValueError('Validation result SHA-256 mismatch')
validation_results_df = pd.read_csv(validation_path)
if validation_results_df['seed'].tolist() != SEEDS:
    raise ValueError('Incomplete five-seed Validation')
ranked = validation_results_df.sort_values(['val_loss', 'val_macro_f1', 'seed'],
                                           ascending=[True, False, True])
if int(ranked.iloc[0]['seed']) != int(selection_lock['deployment_seed']):
    raise ValueError('Deployment seed does not match frozen Validation rule')
for seed in SEEDS:
    path = SOURCE_DIR / f'seed_{seed}' / 'best.weights.h5'
    stage = json.loads((SOURCE_DIR / f'seed_{seed}' / 'stage_selection.json').read_text(encoding='utf-8'))
    actual = file_sha256(path)
    if actual != selection_lock['selected_checkpoint_sha256'][str(seed)]:
        raise ValueError(f'Checkpoint SHA-256 mismatch: seed {seed}')
    if stage['checkpoint_sha256'] != actual or stage['selected_stage'] not in {'stage1', 'stage2'}:
        raise ValueError(f'Selected stage mismatch: seed {seed}')
IMG_SIZE = 256
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = int(config['batch_size'])
HEAD_DROPOUT = float(config['model']['head_dropout'])
print('Source verified:', SOURCE_EXP_ID)
print('Split (12) SHA-256:', selection_lock['dataset_sha256'])
print('Validation-selected seed:', selection_lock['deployment_seed'])
display(validation_results_df)


EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 1%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 3%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 4%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 5%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 7%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 8%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 9%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 11%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 12%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 13%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 15%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 16%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 17%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 19%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b_FULL.zip 20%
EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196

,seed,selected_stage,selected_epoch,val_loss,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,val_balanced_accuracy,prediction_seconds_including_input_pipeline,inference_time_ms_per_image,anthracnose_recall,leaf_miner_recall,false_healthy_count,disease_count,false_healthy_rate,healthy_to_diseased_count,healthy_count,healthy_to_diseased_rate
0,42,stage2,28,0.092685,0.968320,0.968600,0.965607,0.966879,0.965607,0.997690,0.687114,0.957929,0.936330,5,911,0.005488,13,227,0.057269
1,123,stage2,25,0.085312,0.972452,0.971926,0.971137,0.971479,0.971137,0.967312,0.666193,0.951456,0.958801,5,911,0.005488,9,227,0.039648
2,2026,stage2,30,0.081388,0.973140,0.972075,0.971502,0.971762,0.971502,0.978121,0.673637,0.944984,0.962547,6,911,0.006586,10,227,0.044053
3,3407,stage2,25,0.080788,0.974518,0.973862,0.973394,0.973608,0.973394,1.013287,0.697856,0.954693,0.955056,5,911,0.005488,7,227,0.030837
4,7777,stage2,20,0.080343,0.973829,0.972709,0.972383,0.972526,0.972383,0.991366,0.682759,0.944984,0.962547,6,911,0.006586,9,227,0.039648


## 2. Tải và audit Holdout (7)

Nhãn Holdout chỉ được đọc từ đây. Audit xác minh ZIP, giải mã ảnh, đếm lớp và đối chiếu
SHA-256 từng ảnh với Train/Val/Test của Split (12). Mọi ảnh trùng byte làm dừng đánh giá.


In [5]:
"""External classification evaluation for EXP-027.

This notebook has no training code. Holdout labels never select checkpoints or seeds.
"""

import hashlib
import json
import os
import stat
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score


REAL_HOLDOUT_HELPER_VERSION = "006.1"
REAL_HOLDOUT_CMAP = LinearSegmentedColormap.from_list(
    "real_holdout_readable_blues", ["#ffffff", "#dbeafe", "#93c5fd"]
)


def holdout_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def resolve_holdout_archive(archive_name, override=None):
    if override is not None:
        candidate = Path(override)
        if not candidate.is_file():
            raise FileNotFoundError(candidate)
        return candidate
    if RUNTIME == "Colab":
        candidate = DRIVE_PROJECT_DIR / "Dataset" / archive_name
        if candidate.is_file():
            return candidate
    if os.name == "nt":
        candidate = Path(r"D:\Study\KhoaLuanTotNghiep\Dataset") / archive_name
        if candidate.is_file():
            return candidate
    if Path("/kaggle/input").is_dir():
        candidates = sorted(Path("/kaggle/input").rglob(archive_name))
        if len(candidates) == 1:
            return candidates[0]
    raise FileNotFoundError(
        f"Cannot uniquely locate {archive_name}. Set REAL_HOLDOUT_ARCHIVE_OVERRIDE "
        "to the mounted ZIP path; on Colab check the project shortcut and original-folder permissions."
    )


def prepare_real_holdout(archive_path, extraction_dir, class_names, expected_sha256,
                         expected_counts, reference_rows, audit_dir):
    """Verify the frozen archive, labels, decoding and byte overlap with Split (12).

    reference_rows: dictionaries with path/split and optional existing sha256.
    Exact-byte checking does not establish independence of near-duplicate photos.
    """
    archive_path, extraction_dir, audit_dir = map(Path, (archive_path, extraction_dir, audit_dir))
    actual_hash = holdout_sha256(archive_path)
    if actual_hash != expected_sha256.upper():
        raise ValueError(f"Holdout archive changed: {actual_hash}; expected {expected_sha256}.")
    marker = extraction_dir / ".archive_sha256"
    if extraction_dir.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != actual_hash:
            raise FileExistsError(f"Unverified extraction: {extraction_dir}; choose a new directory.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            resolved_root = extraction_dir.resolve()
            for member in archive.infolist():
                target = (extraction_dir / member.filename.replace("\\", "/")).resolve()
                is_symlink = stat.S_ISLNK(member.external_attr >> 16)
                if is_symlink or (target != resolved_root and resolved_root not in target.parents):
                    raise ValueError(f"Unsafe ZIP member: {member.filename}")
            extraction_dir.mkdir(parents=True)
            archive.extractall(extraction_dir)
        marker.write_text(actual_hash, encoding="utf-8")

    dataset_root = extraction_dir / "RL_Cashew_holdout"
    if not dataset_root.is_dir():
        raise FileNotFoundError(dataset_root)
    found_classes = sorted(p.name for p in dataset_root.iterdir() if p.is_dir())
    if found_classes != sorted(expected_counts) or not set(found_classes).issubset(class_names):
        raise ValueError(f"Unexpected holdout class folders: {found_classes}")
    rows = []
    for label, name in enumerate(class_names):
        class_dir = dataset_root / name
        files = sorted(p for p in class_dir.rglob("*") if p.is_file()) if class_dir.is_dir() else []
        if len(files) != expected_counts.get(name, 0):
            raise ValueError(f"Unexpected holdout count for {name}: {len(files)}")
        for image_path in files:
            if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                raise ValueError(f"Unsupported holdout image: {image_path}")
            with Image.open(image_path) as image:
                image.verify()
            with Image.open(image_path) as image:
                image.convert("RGB").load()
            rows.append({"path": str(image_path), "relative_path": image_path.relative_to(dataset_root).as_posix(),
                         "class": name, "label": label, "sha256": holdout_sha256(image_path)})
    manifest = pd.DataFrame(rows)
    if manifest.empty or manifest["sha256"].duplicated().any():
        raise ValueError("Holdout is empty or contains duplicate image bytes.")
    # A marker alone is insufficient: verify cached extraction bytes against the ZIP.
    with zipfile.ZipFile(archive_path) as frozen_archive:
        for row in rows:
            digest = hashlib.sha256()
            with frozen_archive.open("RL_Cashew_holdout/" + row["relative_path"]) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest().upper() != row["sha256"]:
                raise ValueError(f"Cached holdout extraction changed: {row['relative_path']}")
    audit_dir.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(audit_dir / "archive_dataset_manifest.csv", index=False)

    references = list(reference_rows)
    if not references or not {"train", "val", "test"}.issubset({r["split"] for r in references}):
        raise ValueError("Overlap audit needs the full train/val/test Split (12) reference manifest.")
    holdout_hashes = set(manifest["sha256"])
    holdout_path_by_hash = dict(zip(manifest["sha256"], manifest["relative_path"]))
    overlaps = []
    reference_digest = hashlib.sha256()
    for row in references:
        digest = row.get("sha256") or holdout_sha256(row["path"])
        digest = digest.upper()
        identity = row.get("relative_path", str(row["path"]))
        reference_digest.update(f"{row['split']}\0{identity}\0{digest}\n".encode("utf-8"))
        if digest in holdout_hashes:
            overlaps.append({"reference_path": str(row["path"]), "split": row["split"],
                             "holdout_relative_path": holdout_path_by_hash[digest],
                             "sha256": digest})
    # Exact overlap is prohibited for this Split (12) / Holdout (7) evaluation.
    expected_overlap = set()  # Reject any exact Train/Val/Test overlap at runtime.
    observed_overlap = {(row["split"], row["holdout_relative_path"]) for row in overlaps}
    if observed_overlap != expected_overlap:
        raise ValueError(f"Unexpected holdout/Split (12) byte overlaps: {overlaps}")
    evaluated_manifest = manifest[~manifest["sha256"].isin(
        {row["sha256"] for row in overlaps}
    )].reset_index(drop=True)
    evaluated_manifest.to_csv(audit_dir / "dataset_manifest.csv", index=False)
    pd.DataFrame(overlaps).to_csv(audit_dir / "excluded_overlap_images.csv", index=False)
    present = [name for name in class_names if expected_counts.get(name, 0) > 0]
    audit = {"archive_sha256": actual_hash, "archive_name": archive_path.name,
             "archive_manifest_sha256": hashlib.sha256(manifest[["relative_path", "class", "label", "sha256"]]
                  .to_csv(index=False).encode()).hexdigest(),
              "manifest_sha256": hashlib.sha256(evaluated_manifest[["relative_path", "class", "label", "sha256"]]
                 .to_csv(index=False).encode()).hexdigest(),
             "counts": {name: expected_counts.get(name, 0) for name in class_names},
             "images": len(evaluated_manifest), "archive_images": len(manifest),
              "excluded_overlap_images": len(overlaps),
              "evaluated_counts": evaluated_manifest["class"].value_counts().to_dict(),
              "present_classes": present,
             "unsupported_classes": [name for name in class_names if name not in present],
             "reference_images_checked": len(references), "reference_fingerprint": reference_digest.hexdigest(),
             "byte_overlap_count": len(overlaps), "byte_overlaps": overlaps,
             "limitation": "Byte audit only; near-duplicate photos and shared trees/sessions need a separate audit."}
    (audit_dir / "dataset_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print("Real holdout:", len(evaluated_manifest), "evaluated of", len(manifest),
          "archive images; excluded exact Test duplicate:", overlaps)
    print("Present classes:", present)
    print("Not evaluated (no true examples):", audit["unsupported_classes"])
    return evaluated_manifest, audit


def save_real_holdout_seed(manifest, probabilities, class_names, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(manifest), len(class_names)) or not np.isfinite(probabilities).all():
        raise ValueError("Holdout predictions have invalid shape or non-finite values.")
    if (probabilities < 0).any() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):
        raise ValueError("Holdout evaluator expects probabilities, not logits.")
    y_true = manifest["label"].to_numpy(dtype=np.int32)
    y_pred = probabilities.argmax(axis=1)
    present_ids = sorted(set(y_true.tolist()))
    all_ids = list(range(len(class_names)))
    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision_present_classes": float(precision_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "macro_recall_present_classes": float(recall_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "macro_f1_present_classes": float(f1_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "balanced_accuracy_present_classes": float(recall_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "images": len(manifest), "macro_class_count": len(present_ids),
    }
    report = pd.DataFrame(classification_report(y_true, y_pred, labels=all_ids,
        target_names=class_names, output_dict=True, zero_division=0)).transpose()
    per_class = report.loc[class_names, ["precision", "recall", "f1-score", "support"]].copy()
    per_class["evaluated"] = per_class["support"] > 0
    # No evidence for unsupported true classes: don't present placeholder zeros as performance.
    per_class.loc[~per_class["evaluated"], ["precision", "recall", "f1-score"]] = np.nan
    per_class.to_csv(output_dir / "per_class_metrics.csv", index_label="class")
    readable_report = per_class.copy()
    readable_report["scope"] = np.where(readable_report["evaluated"], "true class present", "not evaluated: zero true support")
    readable_report.loc["macro avg (present classes)"] = [
        result["macro_precision_present_classes"], result["macro_recall_present_classes"],
        result["macro_f1_present_classes"], len(manifest), True,
        f"{len(present_ids)} supported true classes only",
    ]
    readable_report.loc["weighted avg"] = [
        report.loc["weighted avg", "precision"], report.loc["weighted avg", "recall"],
        report.loc["weighted avg", "f1-score"], len(manifest), True, "weighted by true support",
    ]
    readable_report.to_csv(output_dir / "classification_report.csv", index_label="class")
    predictions = manifest[["relative_path", "class", "label"]].copy()
    predictions["predicted_label"] = y_pred
    predictions["predicted_class"] = [class_names[i] for i in y_pred]
    predictions["confidence"] = probabilities.max(axis=1)
    predictions["correct"] = y_true == y_pred
    for label, name in enumerate(class_names):
        predictions[f"probability_{name}"] = probabilities[:, label]
    predictions.to_csv(output_dir / "predictions.csv", index=False)
    matrix = confusion_matrix(y_true, y_pred, labels=all_ids)
    for normalized, suffix in ((False, ""), (True, "_normalized")):
        values = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1) if normalized else matrix
        pd.DataFrame(values, index=class_names, columns=class_names).to_csv(output_dir / f"confusion_matrix{suffix}.csv")
        fig, ax = plt.subplots(figsize=(9, 8), facecolor="white")
        ax.set_facecolor("white")
        ax.imshow(values, cmap=REAL_HOLDOUT_CMAP, vmin=0,
                  vmax=1 if normalized else max(int(matrix.max()), 1))
        ax.set_xticks(all_ids, class_names, rotation=40, ha="right", color="black")
        ax.set_yticks(all_ids, class_names, color="black")
        ax.set_xlabel("Predicted class", color="black")
        ax.set_ylabel("True class", color="black")
        ax.set_title("Real holdout" + (" (row normalized)" if normalized else ""), color="black")
        for row in all_ids:
            for col in all_ids:
                text = f"{values[row, col]:.2f}" if normalized else str(int(values[row, col]))
                if normalized and row not in present_ids:
                    text = "N/A"
                ax.text(col, row, text, ha="center", va="center", color="black")
        fig.tight_layout()
        fig.savefig(output_dir / f"confusion_matrix{suffix}.png", dpi=180)
        plt.close(fig)
    (output_dir / "metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


def evaluate_real_holdout(manifest, audit, class_names, seeds, validation_results,
                          checkpoints, load_model, make_dataset, clear_session,
                          output_dir, outputs_are_logits=False):
    """Freeze all checkpoints and validation selection BEFORE any prediction.

    Completed evaluations are read back, not repeated. Interrupted evaluations
    may resume only with the same checkpoint, validation and archive fingerprints.
    """
    output_dir = Path(output_dir)
    seeds = [int(seed) for seed in seeds]
    validation_results = validation_results.copy()
    if validation_results["seed"].tolist() != seeds:
        raise ValueError("Holdout evaluation requires completed, ordered validation for every seed.")
    if not np.isfinite(validation_results[["val_macro_f1", "val_loss"]].to_numpy()).all():
        raise ValueError("Validation selection metrics are incomplete.")
    ranked = validation_results.sort_values(["val_loss", "val_macro_f1", "seed"], ascending=[True, False, True])
    deployment_seed = int(ranked.iloc[0]["seed"])
    checkpoint_hashes = {str(seed): holdout_sha256(checkpoints[seed]) for seed in seeds}
    lock = {"helper_version": REAL_HOLDOUT_HELPER_VERSION, "archive_sha256": audit["archive_sha256"],
            "manifest_sha256": audit["manifest_sha256"],
            "reference_fingerprint": audit["reference_fingerprint"], "seeds": seeds,
            "checkpoint_sha256": checkpoint_hashes, "deployment_seed": deployment_seed,
            # Canonical float precision avoids false lock changes on CSV reload.
            "validation_sha256": hashlib.sha256(validation_results.to_csv(index=False, float_format="%.12g").encode()).hexdigest(),
            "model_classes": class_names, "macro_classes": audit["present_classes"],
            "selection_source": "validation only; lowest val_loss, then macro F1 and seed",
            "outputs_are_logits": outputs_are_logits}
    lock_path = output_dir / "evaluation_lock.json"
    if lock_path.exists():
        if json.loads(lock_path.read_text(encoding="utf-8")) != lock:
            raise ValueError("Holdout lock changed. Do not retune using holdout or overwrite its results.")
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        lock_path.write_text(json.dumps(lock, indent=2), encoding="utf-8")
    aggregate_dir = output_dir / "aggregate"
    aggregate_dir.mkdir(exist_ok=True)
    results_path = aggregate_dir / "real_holdout_results_5seeds.csv"
    if (output_dir / "evaluation_completed.json").exists():
        print("Real holdout already completed; reading frozen results, not running predictions again.")
        return pd.read_csv(results_path)
    result_rows = []
    for seed in seeds:
        seed_dir = output_dir / f"seed_{seed}"
        completed_path = seed_dir / "evaluation_completed.json"
        if completed_path.exists():
            row = json.loads(completed_path.read_text(encoding="utf-8"))
        else:
            clear_session()
            model = load_model(seed)
            dataset = make_dataset(manifest)
            _ = model.predict(dataset.take(1), verbose=0)  # Warm up tracing.
            start = time.perf_counter()
            scores = np.asarray(model.predict(dataset, verbose=0))
            prediction_seconds = time.perf_counter() - start
            if outputs_are_logits:
                exponentials = np.exp(scores - scores.max(axis=1, keepdims=True))
                scores = exponentials / exponentials.sum(axis=1, keepdims=True)
            row = {"seed": seed, **save_real_holdout_seed(manifest, scores, class_names, seed_dir),
                   "prediction_seconds_including_input_pipeline": prediction_seconds,
                   "inference_time_ms_per_image": 1000.0 * prediction_seconds / len(manifest),
                   "fps_including_input_pipeline": len(manifest) / prediction_seconds}
            completed_path.write_text(json.dumps(row, indent=2), encoding="utf-8")
            del model, scores, dataset
            clear_session()
        result_rows.append(row)
    results = pd.DataFrame(result_rows)
    results.to_csv(results_path, index=False)
    metric_columns = [c for c in results if c not in {"seed", "images", "macro_class_count"}]
    results[metric_columns].agg(["mean", "std"]).transpose().to_csv(aggregate_dir / "real_holdout_mean_std.csv")
    per_class = pd.concat([pd.read_csv(output_dir / f"seed_{seed}" / "per_class_metrics.csv")
                          .assign(seed=seed) for seed in seeds], ignore_index=True)
    per_class.groupby("class")[["precision", "recall", "f1-score"]].agg(["mean", "std"]).to_csv(
        aggregate_dir / "per_class_real_holdout_mean_std.csv")
    deployment_result = next(row for row in result_rows if row["seed"] == deployment_seed)
    (aggregate_dir / "deployment_seed_result.json").write_text(json.dumps({
        "selection_source": "validation only, fixed before holdout", "result": deployment_result,
    }, indent=2), encoding="utf-8")
    (output_dir / "evaluation_completed.json").write_text(json.dumps({
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "note": "No model or seed selection on holdout. SD measures training-seed variability, not sampling uncertainty.",
    }, indent=2), encoding="utf-8")
    print("Real holdout results (macro metrics over supported true classes only):")
    print(results)
    return results




In [6]:
holdout_archive = ensure_zip(HOLDOUT_ID, HOLDOUT_NAME, HOLDOUT_BYTES,
                             HOLDOUT_ZIP_OVERRIDE)
holdout_zip_sha256 = holdout_sha256(holdout_archive)
holdout_counts = Counter()
with zipfile.ZipFile(holdout_archive) as frozen:
    members = frozen.namelist()
    if len(members) != len(set(members)): raise ValueError('Duplicate Holdout ZIP members')
    for member in frozen.infolist():
        if member.is_dir(): continue
        parts = member.filename.replace('\\', '/').split('/')
        if len(parts) < 3 or parts[0] != 'RL_Cashew_holdout' or parts[1] not in CLASS_NAMES:
            raise ValueError(f'Unexpected Holdout (7) member: {member.filename}')
        holdout_counts[parts[1]] += 1
if not holdout_counts: raise ValueError('Holdout (7) is empty')
reference_rows = manifest[['split', 'path', 'relative_path', 'sha256']].to_dict('records')
holdout_dir = OUT_DIR / 'real_holdout'
holdout_manifest, holdout_audit = prepare_real_holdout(
    holdout_archive, CACHE_DIR / ('holdout07_' + holdout_zip_sha256[:12]),
    CLASS_NAMES, holdout_zip_sha256, dict(holdout_counts), reference_rows, holdout_dir)
if len(holdout_manifest) != sum(holdout_counts.values()):
    raise ValueError('Holdout (7) count changed after audit')
provenance = {
    'evaluation_id': EVAL_ID, 'source_experiment_id': SOURCE_EXP_ID,
    'source_full_zip_drive_id': SOURCE_FULL_ID, 'source_full_zip_sha256': file_sha256(source_zip),
    'source_config_sha256': selection_lock['config_sha256'],
    'source_validation_lock_sha256': file_sha256(lock_path),
    'split12_drive_id': SPLIT_ID, 'split12_zip_sha256': selection_lock['dataset_sha256'],
    'split12_manifest_sha256': selection_lock['dataset_manifest_sha256'],
    'holdout07_drive_id': HOLDOUT_ID, 'holdout07_zip_sha256': holdout_zip_sha256,
    'holdout07_counts': dict(holdout_counts),
    'selected_checkpoint_sha256': selection_lock['selected_checkpoint_sha256'],
    'deployment_seed': selection_lock['deployment_seed'],
}
provenance_path = OUT_DIR / 'source_provenance.json'
if provenance_path.exists() and json.loads(provenance_path.read_text(encoding='utf-8')) != provenance:
    raise ValueError('Evaluation provenance changed; use a new output directory')
save_json(provenance_path, provenance)
print('Holdout (7):', dict(holdout_counts), 'total', len(holdout_manifest))
display(holdout_manifest.groupby('class').size().rename('images'))


RL_Cashew_holdout (7).zip 2%
RL_Cashew_holdout (7).zip 5%
RL_Cashew_holdout (7).zip 7%
RL_Cashew_holdout (7).zip 9%
RL_Cashew_holdout (7).zip 11%
RL_Cashew_holdout (7).zip 14%
RL_Cashew_holdout (7).zip 16%
RL_Cashew_holdout (7).zip 18%
RL_Cashew_holdout (7).zip 20%
RL_Cashew_holdout (7).zip 23%
RL_Cashew_holdout (7).zip 25%
RL_Cashew_holdout (7).zip 27%
RL_Cashew_holdout (7).zip 29%
RL_Cashew_holdout (7).zip 32%
RL_Cashew_holdout (7).zip 34%
RL_Cashew_holdout (7).zip 36%
RL_Cashew_holdout (7).zip 38%
RL_Cashew_holdout (7).zip 41%
RL_Cashew_holdout (7).zip 43%
RL_Cashew_holdout (7).zip 45%
RL_Cashew_holdout (7).zip 48%
RL_Cashew_holdout (7).zip 50%
RL_Cashew_holdout (7).zip 52%
RL_Cashew_holdout (7).zip 54%
RL_Cashew_holdout (7).zip 57%
RL_Cashew_holdout (7).zip 59%
RL_Cashew_holdout (7).zip 61%
RL_Cashew_holdout (7).zip 63%
RL_Cashew_holdout (7).zip 66%
RL_Cashew_holdout (7).zip 68%
RL_Cashew_holdout (7).zip 70%
RL_Cashew_holdout (7).zip 72%
RL_Cashew_holdout (7).zip 75%
RL_Cashew_hold

,images
class,
anthracnose,46
healthy,13
leaf_miner,36
red_rust,25


## 3. Nạp đúng kiến trúc và đánh giá năm checkpoint

Kết quả bao gồm từng seed, mean/std, từng lớp, confusion matrix và seed triển khai đã chọn
bằng Validation. Không dùng Holdout để điều chỉnh mô hình hay đổi seed.


In [7]:
def set_global_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

BRIGHTNESS_FACTOR = 0.08
BRIGHTNESS_VALUE_RANGE = (0.0, 255.0)
if not hasattr(layers, "RandomBrightness"):
    raise RuntimeError("This Keras runtime lacks RandomBrightness")
AUGMENTATION_CONFIG = {"horizontal_flip": True, "rotation_factor": 0.08,
                       "zoom_factor": 0.10, "contrast_factor": 0.10,
                       "brightness_enabled": True, "brightness_factor": BRIGHTNESS_FACTOR,
                       "brightness_value_range": list(BRIGHTNESS_VALUE_RANGE)}

def densenet_preprocessing(inputs):
    x = layers.Rescaling(1.0 / 255.0, name="densenet_rescale")(inputs)
    return layers.Normalization(mean=[0.485, 0.456, 0.406],
                                variance=[0.229**2, 0.224**2, 0.225**2],
                                axis=-1, name="densenet_preprocess")(x)

def build_transfer_model(seed, weights="imagenet"):
    set_global_seed(seed)
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="raw_rgb")
    augmented = keras.Sequential([
        layers.RandomFlip("horizontal", seed=seed),
        layers.RandomRotation(0.08, seed=seed),
        layers.RandomZoom(0.10, seed=seed),
        layers.RandomContrast(0.10, seed=seed),
        layers.RandomBrightness(factor=BRIGHTNESS_FACTOR,
                                value_range=BRIGHTNESS_VALUE_RANGE, seed=seed + 1100),
    ], name=f"train_only_augmentation_seed_{seed}")(inputs)
    x = densenet_preprocessing(augmented)
    backbone = keras.applications.DenseNet121(include_top=False, weights=weights,
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    backbone.trainable = False
    features = backbone(x, training=False)  # Keep BN moving statistics fixed in both stages.
    x = layers.Dropout(HEAD_DROPOUT, seed=seed)(features)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32",
                           name="disease_output")(x)
    return keras.Model(inputs, outputs, name=f"DenseNet121_Transfer_seed_{seed}"), backbone

def set_backbone_stage(backbone, stage):
    if stage == "stage1":
        backbone.trainable = False
    elif stage == "stage2":
        backbone.trainable = True
        for layer in backbone.layers:
            layer.trainable = (layer.name.startswith("conv5_") and
                               not isinstance(layer, layers.BatchNormalization))
        if not any(layer.trainable and layer.weights for layer in backbone.layers):
            raise RuntimeError("No weighted conv5_* layer was unfrozen")
        if any(layer.trainable for layer in backbone.layers
               if isinstance(layer, layers.BatchNormalization)):
            raise RuntimeError("A backbone BatchNorm layer is trainable")
    else:
        raise ValueError(stage)
    return int(sum(np.prod(v.shape) for v in backbone.trainable_weights))

def decode_image(path, label):
    pixels = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    pixels.set_shape((None, None, 3))
    pixels = tf.image.resize(pixels, (IMG_SIZE, IMG_SIZE), method='bilinear')
    return tf.cast(pixels, tf.float32), label

def make_holdout_dataset(rows):
    dataset = tf.data.Dataset.from_tensor_slices((
        rows['path'].tolist(), rows['label'].to_numpy(dtype=np.int32)))
    return dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE).batch(
        BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def load_selected_model(seed):
    model, backbone = build_transfer_model(seed, weights=None)
    selected = json.loads((SOURCE_DIR / f'seed_{seed}' / 'stage_selection.json').read_text(encoding='utf-8'))
    set_backbone_stage(backbone, selected['selected_stage'])
    model.load_weights(SOURCE_DIR / f'seed_{seed}' / 'best.weights.h5')
    return model

def clear_holdout_session():
    keras.backend.clear_session()
    gc.collect()

checkpoint_paths = {seed: SOURCE_DIR / f'seed_{seed}' / 'best.weights.h5' for seed in SEEDS}
holdout_results_df = evaluate_real_holdout(
    holdout_manifest, holdout_audit, CLASS_NAMES, SEEDS, validation_results_df,
    checkpoint_paths, load_selected_model, make_holdout_dataset,
    clear_holdout_session, holdout_dir, outputs_are_logits=False)
if int(json.loads((holdout_dir / 'evaluation_lock.json').read_text(encoding='utf-8'))['deployment_seed']) != int(selection_lock['deployment_seed']):
    raise ValueError('Holdout evaluation changed the Validation-selected seed')
save_json(holdout_dir / 'interpretation.json', {
    'status': 'EVALUATED_SEPARATELY', 'evaluated_images': len(holdout_manifest),
    'excluded_exact_split_duplicate': 0, 'source': SOURCE_EXP_ID,
    'selection_source': 'Validation only',
    'use': 'external descriptive evaluation over classes present in Holdout (7)',
})
display(holdout_results_df)


Real holdout results (macro metrics over supported true classes only):
   seed  accuracy  macro_precision_present_classes  \
0    42  0.758333                         0.774691   
1   123  0.791667                         0.794210   
2  2026  0.808333                         0.806565   
3  3407  0.808333                         0.832071   
4  7777  0.783333                         0.780183   

   macro_recall_present_classes  macro_f1_present_classes  \
0                      0.680268                  0.705120   
1                      0.701971                  0.726196   
2                      0.743452                  0.764794   
3                      0.775646                  0.796412   
4                      0.716407                  0.734913   

   balanced_accuracy_present_classes  images  macro_class_count  \
0                           0.680268     120                  4   
1                           0.701971     120                  4   
2                           0.743452

,seed,accuracy,macro_precision_present_classes,macro_recall_present_classes,macro_f1_present_classes,balanced_accuracy_present_classes,images,macro_class_count,prediction_seconds_including_input_pipeline,inference_time_ms_per_image,fps_including_input_pipeline
0,42,0.758333,0.774691,0.680268,0.705120,0.680268,120,4,1.378899,11.490829,87.025923
1,123,0.791667,0.794210,0.701971,0.726196,0.701971,120,4,1.327306,11.060885,90.408678
2,2026,0.808333,0.806565,0.743452,0.764794,0.743452,120,4,1.374091,11.450762,87.330433
3,3407,0.808333,0.832071,0.775646,0.796412,0.775646,120,4,1.353999,11.283322,88.626378
4,7777,0.783333,0.780183,0.716407,0.734913,0.716407,120,4,1.367782,11.398185,87.733263


## 4. Đóng gói và tải kết quả lên Drive

ZIP chỉ chứa notebook và kết quả phân tích; checkpoint vẫn nằm trong gói FULL 027 gốc.
Rerun sẽ đối chiếu hash và cập nhật đúng file ZIP trong thư mục evaluation riêng.


In [8]:
NOTEBOOK_SNAPSHOT_ZLIB_B64 = 'eNrFff1v5Mh14L9S0SJo9rhFtaT51LoNaCTNrm5mpImkGe9aEgg2m63mik1ySbakHp0CBAZyuB8MZC+4Hw7B4bJZGEHiGGfDBoybQRDgtPD/oftL7r1XVWQVWezuWe8msrHTJOvj1atX77uqrpc8PwyzpQ12fE0/nXya+PC4NHbT80F8GS112FIwwDf4ebnbxRdjP3cHbu7C6+sbeM7iSephteOlj9i2H2X+np+vrq2ync9eLXfXHrH/+wd2mIRBzqzVtTb7f3/939incTiIJ/DiUfuEOuH//fZvbr+ORuwsuP2affvV3fv/4bHw7t2vE3bvXvTtz8fMG/neeRIHUQ6fb79ho8ndu19GLJxM797/l+jePebdvf9Hl6WTiGG/d+//F1t78JAlVzbbi3O/H8fn7Hx0+/vojOWpG0TY+tdBh3cuPniju/e/iFjm+wOWj/xYhbXDLm7/XrYAEBTAUjvsyM9yWx3qt1/98Td377/xWDSCYURsECCcrH/37rfRGe/18NPNZYSRQ/6z3VfUxdiNgiG0BmDG0NXZ7e8C9uz1ixfs7O79V97HEgTv7t0/Riy/e/dNIEBY7dx/xD55ygZ37/83C7G7CR+qreJ5C8D5eopVoHpIMObMu/3aYzjNUOH2nyO2FYdun1mfvHqNCM2Dsd+22RZ9pw4lXhBh/xX/i01m7gSAC9gVNjYOohHv0IujYXDWYW/cMADSCWLoNvbOO+VAcdTl9Nrs+d27f8vZlzDB37AI8DUWqMhHf/wNG9+9/6W3gshKAwD1jKV37/82KNDN+xRowaGUldg2EG7m52KM22lw4cOobn8HZe7e/xyp7t0fcjZyZWv/E1of3L37VURzA0C7oYCEUAUz8MsJG93+SzRCDJ/CeqgsJS8e+NVltDpnGQXjJE5z1gdIH97vsDOvAxBlozDod1gQd9gXWRx1WJx1WDaa5EEI/07hAeeow94GyTAI8QeU56gYpjEsnjgMfQ9xnzHRwVYME+unSiEAyMdmZAn5zBt/G0e+Ujhxc4RJln0Fj/yreBFNxsmUuRmLEu194kYDeAv/TwbahxyYR5wOw/gSP+ZDpS/lkyh87qduZixh0ydZLnSnfqoW3H01zUdxZA+CLIGPxVD5o7pOBv6QIS6dbOTCIrVwwO0N/o3B3yA4Q9rtydmxRbl2WeQyyEeEGV7ZjhM/slppv9XGIWZ56rtjpUX8G8Yp6+PyYMChApgfK3TH/YG7IYrb8J+B9Zj9+MdsrQs8qd9qtStNlMDZkwTn0KIGFbhSP5+kkSw08q/4L6tdHX/mXvgOEhwNADigG058tT9lcJcpgOvk/lVuYQ17AASQWVSjw2B6JqnvuJkXBL1nbpjBuyAa+FHeW2vjZ1gpQXTWa03y4fLjlgZIkgJTsFpHNMXPYIpbQJBD23EuYGqBoh2nw1r/mQGvyjb4J85x7DDIcicZTbMAFq4z8C8Cz8+sFhRstbGHU2jcv/K9Ca4Lx8P1ACswmoSwppaAuyWTnOTjggt7ba58fJbGb30unvj7DotimA0vSHwGE6+IOG/kRjAngnMf7r8+2NpxQKY6u9tAci2Urts7e4c7eztHIG2Xjw429w6f7RwsPzjc2dk+RMmLkiW5Wl73V588HHb7La2lrf3tHedg583u4e7+HjYoi61373vr/rD7xHvyaPDg4WDt4drqI7c/HHbd9cH9B11v6D/0hl5/vQ//6a4+Wl3vrz154t/Xm0dx5extvtyBpnXYf8Ra9NUGTmWow0e3ur42OL/youefD8ef7jx89Hr6+dWXf5H+tPvy0daXyZfdt4aaTz8/2jmEyg/Xu/D3ZG1NFHn1YvdIgtLagqXqX6IYcEpJrYJCpQUQn73cm3z66uHBszdhnm0+Hj79zNscPZpEg6u3cfb8831R59P9F9v7r8s+Dl44vBtnVCoPSh+yvOjl6d6zy9GTBxdbR9Heo8+/eLHzeC1zguTz129hKbj9Lyu15DDXH3Uf3r/fffCEf94+2H2z4xxtHnyyc+Q8g7I7B7KD5Zfpl9njz58MLl/4V4fPd988f3ox/fzi4i8ef/Z4+FkeiA523mwK9NcmbFl03n20jKUMyAfx6Oy/2Tk42N1GHOyBqGDsI+bbZzZrrcByBP6crwxQ4K68nJLgXbFtu0oJcpDG5vBvTpPzUP/61Yv9zW1nc2/zxeeHu4fO0b5DiIM+jtKJkG4/3T947hzs7x/BW2JvRW/AtoNh7Z0dZM4gSK0284Gt0WfbuxxIZrq1ufXpjrO9ewDNlU2vsNYAleWIlGUHlqsEufvI8VxgBAJkRIehMql1wDCd1M8mQJ4tfOlN0hRAcgZczcF3YlYroNjjc4Q4cbF41sOxAw++QnYZn9NjW+t94fKCVR8Sc0NmrNESsekdlAikBeJ3Ad8PwI3X51srAXDenKuBy0DpaPOEsTvIbP6Obb7aZZPMB03FPfMPg7fIrn0QOvTsZPDCLkXlmQ+Ix2pO5qcoaazMA3nfa43yPMk2VlYuLy/tszg+C303CTKQUeMVd5KPOAmTWI+jcKpJc6Gc8H9QxRD63qSfpDHIsqwsinpD4nrnABko1vFgEvqoQRxbLd7nMvYFCBePNj22O3XNofpXNpAEywkpT8teGCDpF63BJ/GqfVpRRvJ0ugGAZLkLCugAyFgfjD0ELcDJEt+zONBtvbp/5flJzl7SNzDjngFZDHbSNE71VvcK5VTWhJVaFggyKmFQlEpc2iR9YfGFoXUMKrXNidHtozrdWh7jcBNgJfCPaBh+CpSftmuTpuBZmSVUQSs4Q1XUi4GsCmW0PwnCgdLgsJg1D82yFk4sAsgxlqHm0MiUquplCQBvTPapw4l/+IbAhwZBgQLdDtZCqqq3XuqjBheAOtdhDsyCMmYb1oQLnImvgqx3TP+e1pVQGqzVolWAuL1Yh/8qLfe0Xog1OgXGuCrZNqrt48GD+To7FvoPUNi/L/1cqNUg4SwaczAAFuWOiTvDmsr9gdOf5j4gDtGVBgO/h+tA4zHD4hsukyjOTUsF0SHloSzehhWC1lyVLESr2BJWQ1JE4EBAEo+CN7B6YDDwDzFR9me9CrhYEKsLaxZbED9rMyr/QB6C8H2D1gaxB2vYekH2Olrug9jnIxu7uTcqOiNUrSAIIB6w4ZtWZRhiIhJXXRyCvyO913i+Uh9lD5QRX2yEPoNBQx2aq91Br5iyYeCHg6zXgt80ewhSB0hzCzlSNgHWM59PEy9LcC1nm2FIAoxL6rZgZBpwMD8I3zF02TrFCRCwIOb5BwSEf8Jf+B5FO/+G8LVO2/WJq0yMYVK4ZMXegLyHQ7DgOE8actOIC2iYDuxIm44BLIAg4u6jHiu1qhWCTxuZUlQhPqBU7UuFCHtzxiIoGuGiOWyp80OkXXAdpZc2tsuRppY/NVGwWPRl5TlyAxULyb5f+oPA3Y2fgt63LdSYsnbuYyEXBExPwwByOyebDIfBldVCNdkGDS9vaYzoywlnmAYydsbYaY2YG6iwwmQLmASPvZzFY6VmBqpXzzRUi9fqSHhBTowm0TnOa09y3mqLES5gEh/6l8sR0iZO9cCsMYC6cAZaN/BU0UgJnR35V7lDfVvRZAzKeZ4GgID1dr0ZoCfZ0gbjejPn3MPWtfxgyx8gx+3un2vrAeqXOPwTGGrRSHv+4t2NQGtNQj/3ka3CKkWIq1A1rRBcf8USKXtFYOcvkEZO8nL7ARsHGTF2E0AljlI/CV1g0erqrItcbfX9CSZJUxDl/jyz5CO2arPPpAe9CGfkKTqk/84jB/sYgxsNYZQt8sifBeQAj27/KWJz/e8ddq443MtisJZzUG5h7QvfMTnqMRgjHPuluyq/e/+rMlJhs5d3738RkJYlHOakISUjguocQPy7RMQTsrt3/4oRi19BoQjAmAgZwPJRQI+j+O7dHzyWucEHONkfzEEz/4mLABawokXpvqjCfi38WfobcsbMkc1NbpK25kXhJr4q1TTLmZeNfNCE0dq5buHyToMxWvvC2Yk+V9Kfi+lzinlzEPuiRBXalvAWOJIabC+7wIbcM2A9Z6CerihNCneD8wBDZBkVvRHedWAxFDdDq/P+Woetrq13gPGuPeyw9fvdRx32CP7Udc2HI3Xg62ELqzvX+N+blT5CcukHZ6M8s0cPWsgX1e9EmeUI+eBuCjeEDzLN5MpaEW4VkkGSG/4sSJ4hKyyJgkSRm3ojYDEqxMBdMmhWfLHxGZ3MFc0qBIFGRYm54RPgV7wB3b/Gy1rbkyQkE4vH+UTzwBbHfVCQKvwV+bhAXZCB+YptE987xp740Nsbp2SYkOoGE8IBx8rwA6UFyGdEgSx+M5f5tzTIgJGeZyRtA7DOhPJG4V7SD1SIEYzUD4F8LgiUDHQDf2DxEVS7BcBAcpQTx5eCrG4qa3On1MIeKlmb41aZS2griIaxwAjQiuy1btnwuqXfj3RmesXjRSiCf9xj3cXMlN2I1pdke9gE8CvZe80kQbnPR66rtuJlXautgmaACt0GQTQhn+1WOk1y0DrcZBR4PCKRsb5PYbkwixkYf8Fwyg1xLtBlkKKYGEXdFFCRpom0Z4kXRJiwHmsaZ6Egynkh9ZCPgSuIfJFKja9Jj+TS2eTwoaCp7cXJFIGP+19YlSZ51Q4za45z9K4atutT3kBYdU2F40qT7pzZO8Ic11ZJk0zgFVECmKvNERhqv1CXQnvkJbUUWMjJwWN/tVCegF9vW2+pAG6BdgD/vONjdcDSjNUZPokktdv5dRSKqfPAn5IqIzNbONm06nChQgKS8iKg6CQPC2tdadG3mT3KvrBJJpsUwcGBoWvp+j89bokFJDouw2AYZ6yX5m4MYcSpdeaghLv6pedIWPFlbM0A4hiGEmKXfRAifTCheG8tJXmopQJZlBfagCgejEEBiAD4meC9xMoywtqMtqIPd3KGtEHLgWBMsc8ITDBn6Hp5nPLeu3b3McLYgpm9wtml0QdR0WAQAQvRrJh5qMtHvshXKgiMgy3BpcAurJsZypWwAF5sHh7SXFPxlhvlo9T1ojgjB+vId8N8NMWfoe8OHTAz/BSfQJI76SQjvz5A5Hg8lIaFWqc1lJECKAiFIDPTlRe6gDvSP3hhBbrZ2MH2qVFsAX6g9V+fwer6NoFVLfOdYapYUKTvZivUHKhXdeDU9BGFW5JaWAOKf5fcYiYcW5wby/QxzZFWgVGBxmpmEhqTqkImDYSiTJzqksvQrjQmFmq4Vrg9mzLKHLshCAg/Je/JQlgoMs56LBlwYQMmjKULRKNBVKIRF+t1K0MgKCTk8rialOqOfBG6oDXhDzGqm1JdL1uOw8k4ymaPV/G6KEOXbWhroWz6WEDIvbTXPGAs7EP8J8eqN6S6ljUE+gHQyAW1xo2m1sJTUaB2DMOEaWGo2QWUcbaCfoDSKgC2BqsZ7aii6+P56Ds9tfMYFG4vt1qiheqsOuRvqCVkKdlIomKHrBDn3J9KUyHzAVwX+HvWs1od7HYDNA6blBDQsNv1IAzaAVq/H0LgH4xUudSlq0sOXVHejMpdYcS3KqqewZA3sq1KB0bWpWqQiwxQWZwcjEWGJwEeDCuLtwphMQxjZS4hWkhM3HIvJMWia3CI6QLk6SjHUZC2G52Th8bYt01URxlxmUVoAwRmmViVQCVeGjvDVSJ6AnKxsI/652aeH5HafMwJW+Ta4cNpiRr0NnNY7QDm8Lh7KvFC+MDPNVoGoySekuIsSs5E2HZRnHuFKiE4oSiqlDAJC1VH9SVV58ZE5ppriCi96j5Sonfk0tTMD2tOY0ZfU3sBW4VmxMsnbsh69VxSzasjihlWF38GiEp3a7HMjsFmtRDQ9gIO863SXVtdbxsc3WLMOmQ0etBP6r0jtAJunDBergCXnqGQ0IyvORo5deOvtdbNfJgPRWti2mbCu/vyE+dw92eYzQXg8Xd7r186pN2RLowOMkXXE/Webh5tfSprIulLfaaPXTki2Cky1HY2t53tg/1X+68xP2wIBFSWL6yIEVLGII2TGI0AU24Ud6ME/qCWJFUpXcoCMWdYYZ6GprdRLrLlrECnz7uev8pFsgHPj7aMfK39w4RKHs4PlazZ7IhCChiYcCcDwFVDWGRvdPtNVNm2UOzQwO0mv/B4EOPbr27/YWqzTWqs2MmA8a6OjKqMb79h8CMadWjLwL+NWYgBDQICm/oqgA4oeKFv88h5lIOqsguoEgBfBrVsBWZoBfePiI0gSmqqCKTwKnl6+3+gPkb0WHj792MZNlE2o3xAiOTRHPSe0P92rnI/jWCFk5UDJOtxdu0XaXzk4RUmq7bR5GjETVu+9WaEWwFiJhMXyaNhFxNC2l4m0us4USrZ0BkTAkG6GjlsamciDi5UP+0dsmvtRZxpj+jE03ciBDKhQLwQgYLvfauE1gkwNlh7lBmXTPEX7ZEI8++4q4J6VtoEUwO0WwnDCzD63fTQJ2+HP9jCj2M3Uaq+2n0hC++iq0X5lJ2HUDmyxxje9oo2Xc+bpK43dTJQsgETOsUAv8BSHbIWJ+QZA+jS4Gp+w8NV2WSCLhHuVeMv4NkFOqcnFaf8vwc7my8cmVT86c6LVzsHzpudA5HzfrLU7T60V09Mhbdebr6CIg1oshFah/THUoKdgGEDWpxI43VQO8DcQaeP6t4JlDo+WfpoSH/4BA+Dvu8OffHwZN17MBycLAlPS7s+Gkz4kq1/z3tSAPb+ydKfmuS22l27z+4x/AcT3XCJfo97U6BCUiaaVRED0igOwXyUCBKODUs6OHhuxZ+YBefBKkPx59dS4Yy5QkXxMlTTGBLCAKSW32oVtc05acVnbRAHr/eOdnEfApI3beI7WZoxCL5t4NXB/n/a2ToS+u/Jktgmd7IETyr+aoNccICzAI4zHgsigKN8JrSE8hTg2zg5Ocwng+nJyfNR7L6YuNFRnO+djQI/OTkpoG//O4BPIJ0srZyDoR36K+T2hZ6b0m+LRtD7ISKhTU2kZ2Hc1+i3XScz1GjLRinbbHWBMWRg8VWttzoFVngAEJQbIV1PouDLiR9OaUckzMy1CuSNzQ79nGncdPNg61MktGJPh+S5su2TpTwmD/iYNiUOKGsT+dPHLJabUUkdoEJJGn+BCkI2AgyCxknxzzgNzgJQVJaHwAEwR99PyRMVR6A0KN018A+QLBg8dlQeXiCfb34DSw/EG8kymFsh37gbWcm35Rx3nuVeFCdNOUNBNvRTsCMBgvgSnkmZxX5UEuJqzxsegUVMCDNawNkRalSHDXxukBJmSF9EThW6CWf/ioKp4oFTigoHJirTiF3MY+OVERkr5FzkeE/oe8ij7dgnx4ASFN65ArQtExg0iVio8AcAbwcxGWQj2g8IwmiAvbMYWCiI3uVBkRqRjOI8VmPNqhJIlvbM2SoQSn7HhJZdh1kLVqpb9IWzUZfIansmC7/wFVYIRkq3uSbxyZLUmWUyhog7gAavdHHzcZnmfF3p6wYWhJqoDPYXpULooydRYOsBAhXdMCS9gk0UkNXGIDNnqRs9G1y8M3hSTpbIlYKMEFSMIOGOOmWAMyXpDkFSYOx1JM1tBeQNxIsKP6DMG8UxNOEC7V0C6eP+oDid6ujCXV4bhoSFajaRRgnmfKKSPZPyMoA1F+e1ibDFZ8uQRTosck9QM5M5E5jIwr2bG2Y+VOTZWLU5VxIYKFlIJiPA1OMfKasrgJD2LLBo2jMnm47DIMJgP9pY9qGze/hi77nI5gBy4aal4+Z5yn7yE7b6sKmpodoaDFmkRiBJ6LhDjqS/Ee4nLVsoa8JL05J7HWWgqZNU4sBThrqGqBuNSEo+r81kPUvJUEfOoqiL+5H0ZqqpTHwRKfuvlWWi7K0ul1SV6Uv/kZH8iA/UdlZWOAHlSyutNClATaqGWldLHoMSjgywFipTwnVG2vOGk6v3DFTF87IwxboARGfGesN/VrRckcttmTiNoTWtTruM5il6wCLs+3VUMGa5S5WHuLniAjL3WutJJyyUyhjVP9Uz7Ejwd4p8Pz+ajP0UzasZwPFPXCJqFFDdUkGdYKKXMgMl8otmhMZ6snQPdW+JfM7s6bksqW+ZVQej6LXUpb7JhM8KZZdzg6672F4gM9axMRoHTxsHxJfd1pczFqSkEx5pg5FTyQ1jan9Z0OYbLOwwvkT5XrjDQbZ+kZxxfgq/fPkzieSv/jgRvy79Pvy8aWBaDQyLb8NQxkswwSBL2Mw8i6QZ+Xy4c6AsT2KMN9PAqKkWTxG0foimvRhFeQ5EdvDJU1QOaPuHoSdcJ7aboD5pAa5pB9cSuSvUTjvktVEiz1hGmbviWx7rLMp2MyeJs+DKmruH9mSJ6B5b5jR7skSrFV+IZXuyJNWrjao+qQB7o+lsaiIDmrzPUkyvxGFXtmPIXAN/nORTLeBfdntqF3r2wBD4N1JZqYgGGSsax1RSN4gyViruNASyQjJdk/qIbUrl0w1xLw00BFVxvXi4w2mjkmxayiXeGnPPsCueOwWy2a54ueZqZNx8csyKGSUu44kzERFToyNrprdNI3y9O+l5q4lW0K4w/Ti+PK4R52mzi+4HcdUt5rJT+F6z2450NRpTQXQfwM+2OAVIVqZQQmn9YOuVPJJTncMV5twHpYsXKyiPKeGgtCRRM5JEZcoaQvaNVu2VYZ+yZmyjZCWFXTe/64n/Sg2hmQBzo5gKFxUXbsh/YH4PiAxli0CKqEezHTDPiZtIu2jwZoEVvy+8CDzUFlGKHvkhJnRyGMayAIKVnFJWytBl0UmJS21eJM/jeUK021DJZFIoxlAH59npT6U5TglCuJPHVL2j8r7a0lIaF96SqqZVzs4CS1/lHwWSqxsbZTNQkLSaElhSPSvCgK8fCW9DU8VaTQzbsmknfT7VeqwgokNCUu+q5oLW8SAZAyxUvgpF7tnNyUn3WvZJD7w8/AR6ldlVpV1S92+KUQEOdRoxsA45Z4rUL+FU5b8+NJK+fFlsSAYlVski6TcnS2XAp6JHmCj0mA9owaZLrUDgTROe5FYrHHsgO5M0HgX9AHU+JL4c46/KKlxRY+NKAFcRm4WiLVvlS7GN3R345HQF3QCKYc+V2LWs4ubyyECl4biPG4i1hq+tCrYF9hvwCaJPWVFysm/0YEK1F9V2EO++k4W2oqBRdaeisVaConmJOHqhX0VVK7jPXxp1sCALoorT/boiMufgoH2KXhmQQiR1LExBqUqzOmBmuba4PKOsLFUJlYC1zS37V144GZQz4pCCmM3pAMcVIRKPC9tfWruKjcu9ks3WIvsJUyMfXI71UIhWHZ0bTPOilAXINsbPijZJ/gjTmi6rVXJAiUHosqPMhjXw5MKKUOwH1XA4PW3iKHISVLyaE1uNTOlkaT7kdZr6fsfwpw0BuiVCQMDJ0t+YTSINpHVjbpsTLxlzoM7XMdFWaUcvXBYxI75hocjqxTJrql7AUo7fMFPFrJzalJoqSlvtIt+6qQOxJqWjCtsXr8yompROCbXO3AVNr4XfRHRwau6hFPgcVTxn0h9IlJWKmDD9ZXFlIwHpAVUFZy6JoVQo5okwWJ+mSjEavPxtbjUMxgHfjIRlT5aeouzhTAsPC/vYHCMjLzgsKtyaCzafn60ApikSSio7c2W6u2gLlHFFhjTIAV6ScqmWmg77pDLmgz2rzmfO0zE/ESxg3w2lpN1AljBjJRWvWTyURZvXUbHyuBMiw6AYX1NSh0HBXOCPOi/mqw7oKyGEBPFScUGQhtJ7GN0swLUo+W3iY8eYSY7u2CURYTxuWBuG47LqaBFtmMPadICrlpeEaXRWWRd0xr7bD0C7CTCMrcW0edpmNQhdvi2yYMpypmIfZGxr8EAHUWK7mZum7tSqgDrA5MYefKck3If3dWNZK4xyKqHtuZZOLzx/QnGRFw7/CJ3XwBWAzPWO2zYGZD7AOwYEIoLoGchN2gQvNnwTUNRftMy74gfsVtxkuHVLR8uPQUxxJ50CLoDlhXFWAdfOJmPLBWT3VmG4q0AseRz2Vv3lBx8wBEF00BcXnFmVbBCEMD4L8groU4dIvqfa3ULw03YeSme0ipmEdbO+plVH5KGPUxuSm56N3Ss5qpqO6AQDJVKB9gsHo9jyoQXzw1CU5/4XdCZZNarQ1iHtWEGdsZLCIvMfkVXzxHA9I1KA0RHDalfZFWpauAOkTHI0CFjecCUPstKyzAPpKQiBiQe+BjywJ7pB5vPWT1HE813FvW4jRCLLshEcNQvzh4ZluNoMh0wU/eFg6LuhC2rBwCmm9j8YKU0aZYEvTseaRqL0qjZ4oxI5Jd5WggvGhN6moYmF1TEd2sGXVc8sbLxciIja4O0clmeWII9TV72f8kGiU4tAsnFLk9b4MSmrfMlwm4PPDv89XF3mCcTcGuFyGM0ROiXC2NexoomgWd7TPpVNoL2p+mz2YmCmAU9xQo1XEftcO5CqBR4u1soLs5dyMEY8sw3xwlO9/XSI+cjQmG2AkdDwl00gL46U01MuhyM3MvUibDNFLUC1sfjuiGRuzbx3iEh6hemh8VeRN13SYNFVdToqZRHzeMoonxCA+HIEWrxVL6XjAR3oEvMS4RwHkarBbRDi+SwVE9wMDGFfLEJY2mdMLjs5xW0O5nHVtYri5XgBWYAOM71wE5uuGkuVaiaOelrVrGkHYaeWEXpd5Sc3zEzQZK8AVssWTuehjm/fQ3vl4qwBV42lO0wjbsLVzMKS9Ou4mllNWSVmjDGlVn9aoZ7FsGFeYEZmvOgiU3VSdRv2gk4bE2csW+R8BZ9ATsoa0A0XEPOrSHcETrjCxY8D7v0M0A3A2zptaox25BGXFcxZ0x+blEetgRST/yTgXIutD+G7pt6onQ1x7BK8qXMtkslqYB9vCNlqHnUTH1ZKzHSw8s040Gl1f84cAV+JeEUokMLgrU8ngWLmCyLFssQ+ZAo0d5gl14ZTlsf4TwVN3BoiCiXgVpCpw+wFYzBq+DvFvumwc99PBsFYZNaBudPmcVPZBU83UrcdFROi6jm8W4koTZEQBzaoL9tGzA85EaqIvOb4uOHzUM0uwhMaXcR/EuY2nkoexnlmwWs6xdR60mGPYURDUANoJxcs68sRWIy1hlxACWhYRUFrRkFA1Si+LMbrjd2kV9sG1WEX4wD0r8aQ1QXMSG/VjOkri04N5lOFj22aFjPMV3ngnWeW1Bx1Z0Qac0dY734XL0SC8dPRPEjQEiF9PJauCSHTWY0v1MIVET53A3FWxQrWuBgIsoGjQjAuXDcP8tCv+MooTwTeMYykKcgHzcI0G2Ltze5OiS8JVG2Ys46hmdmF+JlmV6jBoaLAiewY2iYQTjfsteGNGVBK0gKyqVZpN6cHK01QCjDhQ3UYzwBSAfRkaW9ls7o5RZkJcngCKBSr7FAtQYyeT5dpLeEdSdrjHHQHZ3aOdAyCcopbTgzf0YcH/y7OX0QK4SAJequPq+fXIXvhfiJoU/lWFRxSbZ/t8eWq5OIuX+HF5NXMLkupcOtbcSrezsr6pb3BHcOpG3Pj7MomY5BssTtwaBc/ZoqcF0k92Jvvpo5wo89ttESltGszBzcXcReZEL21fT3PUt9/6+OK0rY+Iz2XA1MO432682z/YIcC8qWUr23o2RLnlgyUiD/uRfBJz2R4CFtHpBYlPto5NttF2k0nSaWSqitMaQrxXHhQ68WJ3iNYuagElcCrM0LjkN54Jdwya0fPgq5mmn3UFvm5JcAAmXqCCH1WlKY6lRgPbKmpuMYjZdDm9MmQ1E6UoT4/2MHKT8ShQ1UzJg+cAUWKDj/zNTLAAfp0aQh2VXMYV13YBsCPKVusOHWmyB+jM2lIxy8ctO2Fnd5vTIQqt3Mj0QXFQTo6zDNOztGPzVFhlBDrYxDz0VnoMBx+tYB26IU4BaRyPk7RrnoBSnkmSpG6dl0cyVLL5VVW9TGWODXQqeKEE+dkXp8sjfwQnSDivjt0483Y2a7Gm5VkBhFnqn4xBVKNIX/ZQO2TuYXG0Kpsp6HAqZxB8mlyvm5qv3YcDRavzQc2VpldyqbSXxk6+IhtuVEc4cWB3J1bHj0Axn+MEYMhEhOfJHFZH+1VPXyDJweDILFNYNeOyjJkUxhWQD3/ocPBcsgBmIOs/XN7de0MBfWieREwlSjpVMe1JlM1BzL/LudurodJpK8Vh8vwY0V4DFthYjyGjTsj8BJYsaw7KEmAaZDj7NkqD2bTxBk7qctXCqvX3lYWljwuraL1lIy4PAK3stWpPKy2eb/jomfbKqmPdFQ4lF1kP0spOBTyA8G9HQtJnk8in03oGD6ZMg2MBiPbpMYxwAeTtDVvc+N3DORqiG7QHvk1BIvqjsVZdkIhqExd8VnbE6zWEfA3QSwPNBKUoXe3Uj/fo3qAXoVOrGbSKkR7oV03kJIpQwIURKSkaakgfEyqHE622A8ulV9ODZOIDtxRfX60ccKuWSPyuiPldD0VKTVcOQ270TSJVhkSHa5mmMAhZ/vi3LWqDVaMVc5O0cwc1NYPfdBaakK82EJUPfBaqzp7PVfusauvK74FTzErTPtGiEVjcLqwTLhuYdiMIu627mmWS+mPrtfAG9uoSVuQhtziZOfQgoX5AsAw+mAq9rqUAfxTNx2zSYKJ/h6M1iDg6GYBPPYds38xPMUDj/VEdCqMjvNKjocRHhWOtulCHknYgEkwiskWMEDAljl8xo0rdYHR4C3wr5I4EvfRcdjhjSXGsiwGpbiYq17JBh9GgQyt/RXtcYavs4l6r6XaynWpDrt3b25KEAfFYGGT2dVs/KrOZjkTDuj74QTXhkMHmzhJkPhhEPkiWbBSfFbjQSQVRpxbZww8CXRiin9jY6vdbtfusnsmclhZIN1S9DJMZsOsNQQN13u7Md2voDGOBlcKepQWlYXFqgfuIBwWctrEovlAZqMwdLmBAn7XpWNtw2FZr15Y6q2qEGmMP5CR6AgHO8V+uJ+Rb6DhnSP/VnbQCrVQSYNoyHu4Oa3Bdqx3eGqDvEfrcuy7Ihqe5WTuqVkHRUL7bN0A23Cges3Tr+YqABqBXjw3t45VgTtTKM4KrbfneaVsDBieRSRAek1uEpieswgIie8gqGXnFaH4szSeJP2pVUYXjxfPJ5iBa47figu8hu0SBYvhXTH4ivwtvKuNHOfaRq1SrQFaE5svfJHj0asajqr7tA5kpbDoeY5DtZZVtpARhVdIXmGE2YfB+GV8gE8C+Vp1s1f4bZXkn8V5zwcptguOsuSQYNNOco/AFWcq2lF8acljFW34iGcvxNzwteoZUXjiJEfSXiz0J0nlpUsqLjaU2exwG5iPi1eAZcXxlPyA5ws3DURcluvSGabuojI9ATmU4kbnHE+H+W54NCr3ktNZ3P6VrrOYzsVszqzAZOJa06KtJvd7VvG/f88XcD+ec75o5aQ+/R426drCO9jkb37/mnxa5O41+jNdKi9QIkGAHoU7pn6gUwXMSk2+VwKvbtP1XOP+82pT5Rb0DVUO0qVfGDWjT7MuGRNltWvG5Ls5F41JB0J5pE3t1q7yXCEBSeOxQvU7sTaKO6Wq9winhK6mU4ZaeMYQHhG90mrbtDPQwp/G80GorTb7MVvnFwvD03GXXw5T21PfKousFkdSm6/9mHEatbIxUN1K2XgkUBVwnWqOJTyn7Ec9tqpdMKGXNE2lCoA8gKG8YUHdRa5ftDD/4or5Fy5I6LgZj4tLXDegSuOWXra0L4plIDbfGY/eq2/0Lg65U29JtFric/eR02I/YvUVfbyxunaqSgll1juG8h2+hVyfgHb9gD4FB+W5+kiW1QHzeBBYbnqTtghotGefo69OMz+wRnj7mDvM8eAMRGKrOPAb5ESE+aB6fnpLEdHBoLXBdt5s8vstW+LuM/1arA39SHIFebI8bvgnpPF7o9Q6xd2ZtbKCsDa0o/CVSxcN/ei34mzMuzXH0ILiTSd3pBGIwlGpw4BLZXVNG6O4E6tTftVGNu9kdlPz1dtBNha/R0RtrlwICryqHFVKaDAb1oCxWU63rQ3zAlEH1nxjwcYHXW+gNlo9mX5j/tn1ovpNdXFIH6LCuASxlGXUy+/4diK1cuE2pOiE4iKslpt1RwTyhbL87GvgyviwssYFJ/iYTYozC7maVh5dKDkDOX0QziqEHaVB/doAhfPgTQFmrtjK45wuFDJyvsrlAdXvhRHJb+RCiR+8RSs7JfFp8TveMn6dxg9wwcCT+RcMrNts7+7d1wmesf8O9P5zPNY/wpP433nivP/i7H26GVmJQarq9XPlguW+G7Ozu/d/O5YXAnDPHBqvK2C8duRrulRAOS9dZkFit2SewNPd+59H7HzkBgjIN3SvwS9Eln7/7t1voZUyIm+z56Pb3+PZq3SNgJxfvLHg5/BfaOuvJvxmBBjQ+Pb3bHT7L/Br5E6pzH8PRKrBojcMrHbn4Jc2KAJbw2PjQPSTG5JHzhVNlx8Bn8I6i8daugA822UV5ZLexJ7xNR/a537qZjbeAppRah0v7VRLi8tBDnY/+fRobwdUhmebW0f7eE0z3v5X+woS9fWOc7C59wleI2JBmQ5be/DA7uo3h43cDI+7tGA9gMqN9vkBdf+0uGhQP4mJ84IDfqCGjP7RzQrPcRjyqA1xI6+hLd7U5utPXu7sHW0e7e7vOVv7e892PxGpBXEavMUzu0JnGAYJGs4yN1dmXIqbD/ETjny2zXWy9DYGZKpVVruURQ6dAMB59dOc1pQLGIEp9EPuy5Yg1q5nxI+1KfuAPvgOeNoFSN5etHbMk9y+qSauDfACywgIChXaNPZ8ugfNIh+ylgGOWb58/u0DP/Nc9ChYq3YXBBFRDE8m76EHSbSYUjHM4ZXN1Wx60eCeSISkmbOQp/SOu/b9xw86gO77qN7CP92Hi14tRR4QEA3YyNrak3v31rCBtbX7xa8H8GvR1ih2sbxaH1+JMRziVbuK2v4kCEGOoiMWtHAlENZh4pKnnvABR3Ty+4aaHGZgMAp/IXTChHCusIuPliWvEuqw8td6u4A7dS+d9KxfOVtMXBJRtHXofznhkRursnFFTj6t1mew7Cx1IdIRTIWPtjOj6oFYoRatywUr/QzWp8VX5WIVtsTK/aBKJQ+y+MrsfeCyZNomAL4ke+alqMAEJuDqarergnYqpm0oDkpz0F/mqBe9OpqP3bDEcME2rO5i2pXi8mLbghLchLtdaBeGcs+txSNNeLxkItN6JD2Lfz/8CriCqh3aYN5rJOYkjkPum6TtTIYB2IQw5LowFIJP8RD5bj7hIUtZ3LrqFA5UEd/BsPFz30/Y0z02ji/oeHhEepajV5M7rYOI9WNQyOmKLzUdVOGT2/wyLUu9f0slRUuCUzuBIFMaQcxbypVgHTzjJ7jg+wpOQDUZ5mP3inKnaUv6yRIlWa2vGXOPKn+Sp4G8BmPN4b2r7Exh1pws6HJiQW1FonBJrQqhOEeS9emk2ilwXzv4AShVfuNXsVnyscMxXbn+RNyL16NgDF7UVruKYxGq8ENDU2uLNYVCvb4ZgeaOaETW4bNp8NjRB61JrmaRC9Wm2H+GjlmLtJGLBw5deRMNFlpi5DPM8LhRlIe84Y6krKd4U5wmetvmK2Hw0IYqmGhC8ndizc8cd/OBxxUlcS9mxc5DPtx7otFL0BsnEXfq1uKGeFOBAchZIBnxR2e2fwC6Fh3WZslei2YkZCJ+gwDPy2qr2dh8TdRWKqWUT8YWWBXA9QfWBT9LhIdOLzRkFH07Yh7b7bqKSHfDk6ZicfOb9gSpo0+AKVJqCZgqQWxrVfgrcirQudriEGN0BES4J2+dLiIBgnLAxB5zgVM/So06sIVbKvEta4+4Av/vervdAAsdwAwsFm10/qHDDMKljaZsPooHvRaGzvCEolYdr9CghzqFbAieBa9tC5xUcUdpTYXXVaQ3Ud7BRu0kew4xPtjiOiB+ZVcOLDUGJgrS2M+sSpgZ2zrmbvEymb8jXvPriJsPLWkbrrASPeNtI+osdvAWNwcPYApDzL+Ff7KeBHfz9dH+0eu9nbZNt09WQCyvqmxjktTQxyK1qlXMUfJY4WdTssjUsBNPIlFUlwV0brpFS1W1xe2S/363q84WcwKe+qWk7crACVS5bK0PvFu2PvPUZnUWePpNmXMl0nCUOeBaAcLvRwO7MV3nzMPL/HBA+mVoigMViZhvRsB7Pj/0slxWu3/3Ro/laHcxm/dr1eM3DQGgjh6Qoe465vuTO8adFzTWjonIO0aeobZimhIttDNj/5Z2mbJK72p0bEWLvRQp5XPo23QP6/d0L7Nhx5EMKOEGrqa7Yut+5MooA4x+A0viFhUfZEeLQaHeP8HwQQujT683j3a2ncOdV5sH8OvF5xh4LM82E47fDbNbWQ0LFGck0nluDsVUnOJAN2ihWwSiZoa1Klk2COYbPcempZafZFREXmEDgj3z0iDBOKqKWkrbkHka8vyPIFKjx7LZmwZn+Q99ze7q6nwv+H2bffs3t78D4+3s9nf8st2cbsI9V9za4e0/R2wbQ08qU8L8An7bLv73n9zyQlhsRa2fjG7/AfSC2197o4+V9c0u7t79KmLR3bvfjkG9iyUQGGZk3bVH6Er/yhNW44GfTiKW3b3718pFvLTHhrr07t79OmHRCP7JpVcfdSnKhODt56M//oaN797/0tM25gUwwDPu/j41oCzq85QkeL6vPDrjIIpTePng5v8DWJrp2A=='
notebook_path = OUT_DIR / 'code' / 'densenet121_eval_027_holdout07.ipynb'
notebook_path.parent.mkdir(parents=True, exist_ok=True)
notebook_path.write_bytes(zlib.decompress(base64.b64decode(NOTEBOOK_SNAPSHOT_ZLIB_B64)))
save_json(OUT_DIR / 'environment.json', {
    'python': sys.version, 'tensorflow': tf.__version__, 'keras': getattr(keras, '__version__', 'tf.keras'),
    'gpu': [str(x) for x in tf.config.list_physical_devices('GPU')],
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
})
archive_path = OUT_DIR.parent / (EVAL_ID + '_ANALYSIS.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(OUT_DIR.rglob('*')):
        if path.is_file() and path.suffix.lower() not in {'.h5', '.keras'}:
            archive.write(path, arcname=f'{EVAL_ID}/{path.relative_to(OUT_DIR).as_posix()}')
print('ANALYSIS ZIP:', archive_path, archive_path.stat().st_size, 'SHA-256:', file_sha256(archive_path))

if UPLOAD_ANALYSIS_TO_DRIVE:
    service = get_drive_service('https://www.googleapis.com/auth/drive')
    query = (f"name = '{EVAL_ID}' and '{DRIVE_TARGET_FOLDER_ID}' in parents "
             "and mimeType = 'application/vnd.google-apps.folder' and trashed = false")
    found = service.files().list(q=query, fields='nextPageToken,files(id,name,mimeType)',
                                 pageSize=100, supportsAllDrives=True,
                                 includeItemsFromAllDrives=True).execute()
    folders = found.get('files', [])
    if found.get('nextPageToken') or len(folders) > 1:
        raise RuntimeError('Ambiguous Drive evaluation folder')
    folder_id = folders[0]['id'] if folders else service.files().create(
        body={'name': EVAL_ID, 'mimeType': 'application/vnd.google-apps.folder',
              'parents': [DRIVE_TARGET_FOLDER_ID]}, fields='id',
        supportsAllDrives=True).execute()['id']
    query = (f"name = '{archive_path.name}' and '{folder_id}' in parents and trashed = false")
    found = service.files().list(q=query, fields='nextPageToken,files(id,name,size,md5Checksum)',
                                 pageSize=100, supportsAllDrives=True,
                                 includeItemsFromAllDrives=True).execute()
    files = found.get('files', [])
    if found.get('nextPageToken') or len(files) > 1:
        raise RuntimeError('Ambiguous Drive ANALYSIS ZIP')
    from googleapiclient.http import MediaFileUpload
    media = MediaFileUpload(str(archive_path), mimetype='application/zip', resumable=True)
    if files:
        uploaded = service.files().update(fileId=files[0]['id'], media_body=media,
                                          fields='id,name,size,md5Checksum,parents',
                                          supportsAllDrives=True).execute()
    else:
        uploaded = service.files().create(
            body={'name': archive_path.name, 'parents': [folder_id]}, media_body=media,
            fields='id,name,size,md5Checksum,parents', supportsAllDrives=True).execute()
    if (int(uploaded['size']) != archive_path.stat().st_size or
            uploaded.get('md5Checksum') != file_md5(archive_path) or
            folder_id not in uploaded.get('parents', [])):
        raise RuntimeError('Drive upload verification failed')
    print('Drive results:', 'https://drive.google.com/file/d/' + uploaded['id'] + '/view')


ANALYSIS ZIP: /content/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-027-256px-3e196f0b-HOLDOUT07-EVAL_ANALYSIS.zip 780676 SHA-256: 51160f14857c15d6eddd63454fb904c8b3c7a7f66b7c09bbad8dee072fc267dc


Drive results: https://drive.google.com/file/d/1GuRmtGpmV7PvoQJk5AsZMh_OxefU4xFI/view
